In [ ]:
!pip install pytorch-lightning -q
!pip install clearml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.0/823.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.9/960.9 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/PulpSieveProject')

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from clearml import Task

from config import HybridConfig
from model import PulpSieveModel
!ls "/content/drive/MyDrive/PulpSieveProject"

calibration_module.py  hybrid_block.py	mlp_module.py	       __pycache__
config.py	       inference.py	model.py	       train.py
data_preprocessor.pkl  mamba_module.py	preprocessing.py       uncertainty_module.py
hparams.yaml	       mha_module.py	pulp_sieve_model.ckpt  Телеметрия.xlsx


In [ ]:
from clearml import Task
import os

def setup_clearml_colab(project_name="PulpSieveProject", task_name="PulpSieveModelTraining"):
    """
    Простая инициализация задачи ClearML в Google Colab.

    Args:
        project_name (str): Название проекта в ClearML.
        task_name (str): Название задачи в ClearML.

    Returns:
        Task: Инициализированная задача ClearML.
    """
    # Учетные данные ClearML
    clearml_api_key = "OJ1JOQBI0QOCJE01RNJBZX5TDUSN14"
    clearml_api_secret = "pwN-3Dxw8K6Vt0Cuxt9M0qn_3qjqzttYtH3dS02ceM_3mjVZh6pInUgWPmcKo_8Aaoo"
    clearml_api_host = "https://api.clear.ml"

    # Пробуем создать конфигурационный файл
    config_content = f"""
    api {{
        api_server: {clearml_api_host}
        web_server: https://app.clear.ml
        files_server: https://files.clear.ml
        credentials {{
            "access_key" = "{clearml_api_key}"
            "secret_key" = "{clearml_api_secret}"
        }}
    }}
    """
    try:
        with open(os.path.expanduser("~/.clearml.conf"), "w") as f:
            f.write(config_content)
        print("Конфигурационный файл ClearML успешно создан.")
    except Exception as e:
        print(f"Ошибка при создании конфигурационного файла: {e}")
        print("Попробуем инициализировать задачу с учетными данными напрямую...")

    # Инициализация задачи ClearML
    print(f"Инициализация задачи ClearML: project={project_name}, task={task_name}")
    try:
        task = Task.init(
            project_name=project_name,
            task_name=task_name,
            auto_connect_frameworks=True
        )
    except Exception as e:
        print(f"Ошибка при инициализации задачи: {e}")
        print("Попробуем передать учетные данные напрямую...")
        Task.set_credentials(
            api_host=clearml_api_host,
            web_host="https://app.clear.ml",
            files_host="https://files.clear.ml",
            key=clearml_api_key,
            secret=clearml_api_secret
        )
        task = Task.init(
            project_name=project_name,
            task_name=task_name,
            auto_connect_frameworks=True
        )

    if task is None:
        raise ValueError("Не удалось инициализировать задачу ClearML. Проверьте учетные данные и подключение.")

    print("ClearML успешно инициализирован.")
    print(f"URL задачи: {task.get_output_log_web_page()}")
    return task

# Выполняем инициализацию ClearML
if __name__ == "__main__":
    clearml_task = setup_clearml_colab(
        project_name="PulpSieveProject",
        task_name="PulpSieveModelTraining"
    )

Конфигурационный файл ClearML успешно создан.
Инициализация задачи ClearML: project=PulpSieveProject, task=PulpSieveModelTraining
Ошибка при инициализации задачи: It seems ClearML is not configured on this machine!
To get started with ClearML, setup your own 'clearml-server' or create a free account at https://app.clear.ml
Setup instructions can be found here: https://clear.ml/docs
Попробуем передать учетные данные напрямую...
ClearML Task: created new task id=e5952befa1af46db9519a23d9c15c95a
2025-03-30 18:51:12,558 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/d93f0fdec40141e4bab333af9b398cd9/experiments/e5952befa1af46db9519a23d9c15c95a/output/log
ClearML успешно инициализирован.
URL задачи: https://app.clear.ml/projects/d93f0fdec40141e4bab333af9b398cd9/experiments/e5952befa1af46db9519a23d9c15c95a/output/log


In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.preprocessing import QuantileTransformer  # Заменяем StandardScaler на QuantileTransformer
from sklearn.metrics import mean_squared_error
import pytorch_lightning as pl
import joblib

class DataPreprocessor:
    def __init__(self):
        # Инициализируем QuantileTransformer для признаков и целей
        # output_distribution='normal' делает распределение данных ближе к нормальному
        self.scaler_features = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42)
        self.scaler_targets = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42)
        self.feature_cols = None
        self.target_cols = None

    def fit(self, df, feature_cols, target_cols):
        self.feature_cols = feature_cols
        self.target_cols = target_cols
        if len(self.feature_cols) > 0:
            self.scaler_features.fit(df[self.feature_cols])
        if len(self.target_cols) > 0:
            self.scaler_targets.fit(df[self.target_cols])

    def transform(self, df):
        df = df.copy()
        # Интерполяция и заполнение пропусков остаются без изменений
        df = df.interpolate(method='linear')
        df = df.fillna(df.mean(numeric_only=True))
        df = df.fillna(0)

        if len(self.feature_cols) > 0:
            df[self.feature_cols] = self.scaler_features.transform(df[self.feature_cols])
            # После QuantileTransformer данные уже нормализованы, но мы можем дополнительно обрезать выбросы
            df[self.feature_cols] = df[self.feature_cols].clip(lower=-5, upper=5)
        if len(self.target_cols) > 0:
            df[self.target_cols] = self.scaler_targets.transform(df[self.target_cols])
            df[self.target_cols] = df[self.target_cols].clip(lower=-5, upper=5)

        if df.isna().any().any():
            raise ValueError("После обработки в данных остались NaN")
        if np.isinf(df.select_dtypes(include=np.number)).any().any():
            raise ValueError("После обработки в данных остались Inf")
        return df

    def fit_transform(self, df, feature_cols, target_cols):
        self.fit(df, feature_cols, target_cols)
        return self.transform(df)

    def inverse_transform_targets(self, targets):
        return self.scaler_targets.inverse_transform(targets)


class PulpSieveDataset(Dataset):
    def __init__(self, telemetry_data, lab_data=None, targets=None, seq_len=60):
        self.telemetry_data = telemetry_data
        self.lab_data = lab_data
        self.targets = targets
        self.seq_len = seq_len

        print(f"Telemetry data shape in dataset: {telemetry_data.shape}")
        if telemetry_data.isna().any().any():
            print("Warning: telemetry_data содержит NaN")
        if np.isinf(telemetry_data).any().any():
            print("Warning: telemetry_data содержит Inf")

        if self.lab_data is not None:
            print(f"Lab data shape in dataset: {lab_data.shape}")
            if lab_data.isna().any().any():
                print("Warning: lab_data содержит NaN")
            if np.isinf(lab_data).any().any():
                print("Warning: lab_data содержит Inf")

        if self.targets is not None:
            print(f"Targets shape in dataset: {targets.shape}")
            if targets.isna().any().any():
                print("Warning: targets содержит NaN")
            if np.isinf(targets).any().any():
                print("Warning: targets содержит Inf")

        self.timestamps = telemetry_data.index.unique()
        if len(self.timestamps) < self.seq_len:
            raise ValueError(f"Количество временных меток ({len(self.timestamps)}) меньше длины последовательности ({self.seq_len})")

        valid_indices = [i for i in range(len(self.timestamps)) if i >= seq_len - 1]
        if not valid_indices:
            raise ValueError("Нет валидных индексов для последовательностей. Проверьте длину данных и seq_len.")
        self.valid_indices = valid_indices

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        ts_idx = self.valid_indices[idx]
        try:
            telemetry_seq = []
            for i in range(ts_idx - self.seq_len + 1, ts_idx + 1):
                telemetry_seq.append(self.telemetry_data.iloc[i].values)
            telemetry_tensor = torch.tensor(np.array(telemetry_seq), dtype=torch.float32)

            if torch.isnan(telemetry_tensor).any() or torch.isinf(telemetry_tensor).any():
                print(f"Warning: telemetry_tensor at idx {idx} содержит NaN или Inf: {telemetry_tensor}")
                telemetry_tensor = torch.where(
                    torch.isnan(telemetry_tensor) | torch.isinf(telemetry_tensor),
                    torch.zeros_like(telemetry_tensor),
                    telemetry_tensor
                )

            if self.lab_data is not None:
                lab_tensor = torch.tensor(self.lab_data.iloc[ts_idx].values, dtype=torch.float32)
                if torch.isnan(lab_tensor).any() or torch.isinf(lab_tensor).any():
                    print(f"Warning: lab_tensor at idx {idx} содержит NaN или Inf: {lab_tensor}")
                    lab_tensor = torch.where(
                        torch.isnan(lab_tensor) | torch.isinf(lab_tensor),
                        torch.zeros_like(lab_tensor),
                        lab_tensor
                    )
            else:
                lab_tensor = torch.zeros(1, dtype=torch.float32)

            if self.targets is not None:
                target_tensor = torch.tensor(self.targets.iloc[ts_idx].values, dtype=torch.float32)
                if torch.isnan(target_tensor).any() or torch.isinf(target_tensor).any():
                    print(f"Warning: target_tensor at idx {idx} содержит NaN или Inf: {target_tensor}")
                    target_tensor = torch.where(
                        torch.isnan(target_tensor) | torch.isinf(target_tensor),
                        torch.zeros_like(target_tensor),
                        target_tensor
                    )
                return telemetry_tensor, lab_tensor, target_tensor
            else:
                return telemetry_tensor, lab_tensor

        except IndexError as e:
            raise IndexError(f"Ошибка доступа к данным по индексу {ts_idx}: {str(e)}. Проверьте индексы и данные.")
        except Exception as e:
            raise Exception(f"Неизвестная ошибка при получении элемента с индексом {idx}: {str(e)}")


def prepare_data(telemetry_path, lab_path=None, test_size=0.2, val_size=0.1, random_state=42, clearml_task=None):
    try:
        telemetry_data = pd.read_excel(telemetry_path)
        print(f"Исходная форма телеметрии: {telemetry_data.shape}, столбцы: {telemetry_data.columns.tolist()}")

        if 'Время' not in telemetry_data.columns:
            raise ValueError("Столбец 'Время' отсутствует в телеметрических данных")
        telemetry_data['Время'] = pd.to_datetime(telemetry_data['Время'], errors='coerce')
        if telemetry_data['Время'].isna().any():
            raise ValueError("Некоторые значения в столбце 'Время' не удалось преобразовать в datetime")
        telemetry_data.set_index('Время', inplace=True)
        print(f"После установки индекса (телеметрия): {telemetry_data.shape}")

        if lab_path is not None:
            lab_data = pd.read_excel(lab_path)
            print(f"Исходная форма лабораторных данных: {lab_data.shape}, столбцы: {lab_data.columns.tolist()}")
            if 'Время' not in lab_data.columns:
                raise ValueError("Столбец 'Время' отсутствует в лабораторных данных")
            lab_data['Время'] = pd.to_datetime(lab_data['Время'], errors='coerce')
            if lab_data['Время'].isna().any():
                raise ValueError("Некоторые значения в столбце 'Время' (лабораторные данные) не удалось преобразовать в datetime")
            lab_data.set_index('Время', inplace=True)
            print(f"После установки индекса (лабораторные данные): {lab_data.shape}")
        else:
            lab_data = None

        if not all(col in telemetry_data.columns for col in ['Гранулометрия 1', 'Гранулометрия 2']):
            raise ValueError("Один или оба столбца 'Гранулометрия 1', 'Гранулометрия 2' отсутствуют в телеметрических данных")
        targets = telemetry_data[['Гранулометрия 1', 'Гранулометрия 2']].copy()

        telemetry_data = telemetry_data.drop(['Гранулометрия 1', 'Гранулометрия 2'], axis=1)
        print(f"После удаления целей: {telemetry_data.shape}, столбцы: {telemetry_data.columns.tolist()}")

        common_timestamps = telemetry_data.index
        if lab_data is not None:
            common_timestamps = common_timestamps.intersection(lab_data.index)
            lab_data = lab_data.loc[common_timestamps]
        targets = targets.loc[common_timestamps]
        telemetry_data = telemetry_data.loc[common_timestamps]
        print(f"После синхронизации индексов: телеметрия {telemetry_data.shape}, цели {targets.shape}")

        data_preprocessor = DataPreprocessor()
        feature_cols = telemetry_data.columns.tolist()
        target_cols = targets.columns.tolist()

        telemetry_data = data_preprocessor.fit_transform(
            telemetry_data,
            feature_cols=feature_cols,
            target_cols=[]
        )
        targets = data_preprocessor.fit_transform(
            targets,
            feature_cols=[],
            target_cols=target_cols
        )

        if telemetry_data.isna().any().any() or np.isinf(telemetry_data).any().any():
            raise ValueError("После предобработки в telemetry_data остались NaN или Inf")
        if targets.isna().any().any() or np.isinf(targets).any().any():
            raise ValueError("После предобработки в targets остались NaN или Inf")
        if lab_data is not None and (lab_data.isna().any().any() or np.isinf(lab_data).any().any()):
            raise ValueError("После предобработки в lab_data остались NaN или Inf")

        timestamps = telemetry_data.index.unique()
        n_samples = len(timestamps)
        if n_samples == 0:
            raise ValueError("После синхронизации индексов данные пусты")

        test_idx = int(n_samples * (1 - test_size))
        val_idx = int(test_idx * (1 - val_size))
        if val_idx <= 0 or test_idx <= val_idx or n_samples <= test_idx:
            raise ValueError(f"Некорректные размеры выборок: n_samples={n_samples}, val_idx={val_idx}, test_idx={test_idx}")

        train_timestamps = timestamps[:val_idx]
        val_timestamps = timestamps[val_idx:test_idx]
        test_timestamps = timestamps[test_idx:]
        print(f"Размеры выборок: train={len(train_timestamps)}, val={len(val_timestamps)}, test={len(test_timestamps)}")

        train_dataset = PulpSieveDataset(
            telemetry_data.loc[train_timestamps],
            lab_data.loc[train_timestamps] if lab_data is not None else None,
            targets.loc[train_timestamps]
        )

        val_dataset = PulpSieveDataset(
            telemetry_data.loc[val_timestamps],
            lab_data.loc[val_timestamps] if lab_data is not None else None,
            targets.loc[val_timestamps]
        )

        test_dataset = PulpSieveDataset(
            telemetry_data.loc[test_timestamps],
            lab_data.loc[test_timestamps] if lab_data is not None else None,
            targets.loc[test_timestamps]
        )

        # Сохраняем data_preprocessor на Google Drive
        preprocessor_path = '/content/drive/MyDrive/PulpSieveProject/data_preprocessor.pkl'
        joblib.dump(data_preprocessor, preprocessor_path)
        print(f"DataPreprocessor сохранён в '{preprocessor_path}'")
        if clearml_task:
            clearml_task.upload_artifact(name="data_preprocessor", artifact_object=preprocessor_path)

        return train_dataset, val_dataset, test_dataset, data_preprocessor

    except FileNotFoundError as e:
        raise FileNotFoundError(f"Ошибка загрузки файла: {str(e)}")
    except Exception as e:
        raise Exception(f"Ошибка при подготовке данных: {str(e)}")


def main(resume_training=False, checkpoint_path='/content/drive/MyDrive/PulpSieveProject/pulp_sieve_model.ckpt', clearml_task=None):
    # Логируем гиперпараметры через ClearML
    if clearml_task:
        params = {
            "batch_size": 256,
            "max_epochs": 10,
            "gradient_clip_val": 1.0,
            "resume_training": resume_training,
            "checkpoint_path": checkpoint_path
        }
        clearml_task.connect(params)

    pl.seed_everything(42)

    # Подготовка данных
    train_dataset, val_dataset, test_dataset, data_preprocessor = prepare_data(
        telemetry_path='/content/drive/MyDrive/PulpSieveProject/Телеметрия.xlsx',
        lab_path=None,
        clearml_task=clearml_task
    )

    # Создаём загрузчики данных
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

    # Определяем размерность входных данных
    input_dim = len(train_dataset[0][0][0])
    print(f"Input dim from dataset: {input_dim}")

    # Настраиваем конфигурацию модели
    config = HybridConfig(
        input_dim=input_dim,
        output_dim=len(train_dataset[0][2]),
        lab_feature_dim=len(train_dataset[0][1])
    )

    # Логируем конфигурацию модели через ClearML
    if clearml_task:
        clearml_task.connect(config.__dict__, name="model_config")

    # Инициализируем или загружаем модель
    if resume_training and checkpoint_path:
        print(f"Загрузка модели из {checkpoint_path}...")
        model = PulpSieveModel.load_from_checkpoint(checkpoint_path, config=config)
    else:
        print("Создание новой модели...")
        model = PulpSieveModel(config=config)

    # Настраиваем callback для сохранения лучших моделей
    checkpoint_callback = pl.callbacks.ModelCheckpoint(
        monitor='val_rmse',
        mode='min',
        save_top_k=3,
        filename='{epoch}-{val_rmse:.4f}',
        verbose=True,
        dirpath='/content/drive/MyDrive/PulpSieveProject/checkpoints/',
        auto_insert_metric_name=False
    )

    # Настраиваем callback для ранней остановки
    early_stop_callback = pl.callbacks.EarlyStopping(
        monitor='val_rmse',
        patience=10,
        mode='min',
        verbose=True
    )

    # Инициализируем тренер без логгера ClearMLLogger
    trainer = pl.Trainer(
        max_epochs=10,
        callbacks=[checkpoint_callback, early_stop_callback],
        accelerator='auto',
        devices=1,
        precision=32,
        log_every_n_steps=1,
        gradient_clip_val=1.0
    )

    # Обучение модели (продолжение или с нуля)
    print("Запуск обучения...")
    trainer.fit(model, train_loader, val_loader)

    # Тестирование модели
    print("Запуск тестирования...")
    trainer.test(model, test_loader)

    # Сохранение модели после обучения на Google Drive
    updated_checkpoint_path = '/content/drive/MyDrive/PulpSieveProject/pulp_sieve_model_updated.ckpt'
    trainer.save_checkpoint(updated_checkpoint_path)
    print(f"Модель сохранена в '{updated_checkpoint_path}'")
    if clearml_task:
        clearml_task.upload_artifact(name="updated_model", artifact_object=updated_checkpoint_path)

    # Выводим предсказания и метрики в исходных единицах
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch in test_loader:
            telemetry_data, lab_data, target = batch
            if model.config.estimate_uncertainty:
                mean, var = model(telemetry_data, lab_data)
                pred = mean
            else:
                pred = model(telemetry_data, lab_data)

            pred_np = pred.cpu().numpy()
            target_np = target.cpu().numpy()
            all_predictions.append(pred_np)
            all_targets.append(target_np)

    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    pred_original = data_preprocessor.inverse_transform_targets(all_predictions)
    target_original = data_preprocessor.inverse_transform_targets(all_targets)

    print("\nПримеры предсказаний в исходных единицах:")
    for i in range(min(5, len(pred_original))):
        print(f"Пример {i+1}: Предсказание = {pred_original[i]}, Истинное значение = {target_original[i]}")

    rmse_original = np.sqrt(mean_squared_error(target_original, pred_original))
    print(f"\nRMSE в исходных единицах на всём тестовом наборе: {rmse_original}")

    # Логируем RMSE в ClearML напрямую через clearml_task
    if clearml_task:
        clearml_task.get_logger().report_scalar(
            title="Metrics",
            series="test_rmse_original",
            value=rmse_original,
            iteration=0
        )

    # Закрываем задачу ClearML
    if clearml_task:
        clearml_task.close()


if __name__ == "__main__":
    # Сначала инициализируем ClearML
    clearml_task = setup_clearml_colab(
        project_name="PulpSieveProject",
        task_name="PulpSieveModelTraining"
    )

    # Запускаем основной код с ClearML
    main(resume_training=True, checkpoint_path='/content/drive/MyDrive/PulpSieveProject/pulp_sieve_model.ckpt', clearml_task=clearml_task)

Конфигурационный файл ClearML успешно создан.
Инициализация задачи ClearML: project=PulpSieveProject, task=PulpSieveModelTraining
ClearML успешно инициализирован.
URL задачи: https://app.clear.ml/projects/d93f0fdec40141e4bab333af9b398cd9/experiments/e5952befa1af46db9519a23d9c15c95a/output/log


INFO:lightning_fabric.utilities.seed:Seed set to 42


Исходная форма телеметрии: (259883, 33), столбцы: ['Время', 'Мощность МПСИ 1', 'Мощность МПСИ 2', 'Мощность МШЦ 1', 'Мощность МШЦ 2', 'Ток МПСИ 1', 'Ток МПСИ 2', 'Ток МШЦ 1', 'Ток МШЦ 2', 'Питание МПСИ 1', 'Питание МПСИ 2', 'Возврат руды МПСИ 1', 'Возврат руды МПСИ 2', 'Расход воды МПСИ 1 PV', 'Расход воды МПСИ 2 PV', 'Расход воды МПСИ 1 SP', 'Расход воды МПСИ 2 SP', 'Расход воды МПСИ 1 CV', 'Расход воды МПСИ 2 CV', 'факт соотношение руда/вода МПСИ 1', 'факт соотношение руда/вода МПСИ 2', 'Давление на подшипник МПСИ 1 загрузка', 'Давление на подшипник МПСИ 2 загрузка', 'Давление на подшипник МПСИ 1 разгрузка', 'Давление на подшипник МПСИ 2 разгрузка', 'Расход оборотной воды 1', 'Расход оборотной воды 2', 'pH оборотной воды', 't оборотной воды', 'Гранулометрия 1', 'Гранулометрия 2', 'Поток 1', 'Поток 2']
После установки индекса (телеметрия): (259883, 32)
После удаления целей: (259883, 30), столбцы: ['Мощность МПСИ 1', 'Мощность МПСИ 2', 'Мощность МШЦ 1', 'Мощность МШЦ 2', 'Ток МПСИ 1', 

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Запуск обучения...


...

Запуск тестирования...


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.6656668186187744     │
│         test_rmse         │    0.8914428949356079     │
└───────────────────────────┴───────────────────────────┘

Модель сохранена в '/content/drive/MyDrive/PulpSieveProject/pulp_sieve_model_updated.ckpt'


KeyboardInterrupt: 

In [ ]:
model = PulpSieveModel.load_from_checkpoint('/content/drive/MyDrive/PulpSieveProject/pulp_sieve_model_updated.ckpt')

In [ ]:
import torch
import numpy as np
from sklearn.metrics import mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm  # Для отслеживания прогресса

# Подготовка данных
train_dataset, val_dataset, test_dataset, data_preprocessor = prepare_data(
    telemetry_path='/content/drive/My Drive/PulpSieveProject/Телеметрия.xlsx',
    lab_path=None
)

# Создание test_loader
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# Определяем устройство: GPU, если доступно, иначе CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

# Переносим модель на выбранное устройство
model.to(device)
model.eval()  # Переводим модель в режим оценки

all_predictions = []  # Список для хранения всех предсказаний
all_targets = []  # Список для хранения всех истинных значений

# Отключаем вычисление градиентов для экономии памяти и ускорения
with torch.no_grad():
    # Добавляем прогресс-бар для отслеживания выполнения
    for batch in tqdm(test_loader, desc="Обработка батчей"):
        # Переносим данные батча на GPU
        telemetry_data, lab_data, target = [b.to(device) for b in batch]

        # Делаем предсказание
        if model.config.estimate_uncertainty:
            mean, var = model(telemetry_data, lab_data)  # Предсказания с неопределённостью
            pred = mean  # Берём среднее значение как предсказание
        else:
            pred = model(telemetry_data, lab_data)  # Обычные предсказания

        # Преобразуем тензоры в NumPy, перенося их обратно на CPU
        pred_np = pred.cpu().numpy()
        target_np = target.cpu().numpy()

        # Добавляем предсказания и истинные значения в списки
        all_predictions.append(pred_np)
        all_targets.append(target_np)

# Объединяем все предсказания и истинные значения в один массив
all_predictions = np.concatenate(all_predictions, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

# Выполняем обратное преобразование в исходные единицы
pred_original = data_preprocessor.inverse_transform_targets(all_predictions)
target_original = data_preprocessor.inverse_transform_targets(all_targets)

# Выводим первые 3 примера предсказаний и истинных значений
print("\nПримеры предсказаний в исходных единицах:")
for i in range(min(5, len(pred_original))):
    print(f"Пример {i+1}: Предсказание = {pred_original[i]}, Истинное значение = {target_original[i]}")

# Вычисляем RMSE в исходных единицах для всего тестового набора
rmse_original = np.sqrt(mean_squared_error(target_original, pred_original))
print(f"\nRMSE в исходных единицах на всём тестовом наборе: {rmse_original}")

if clearml_task:
  clearml_task.get_logger().report_scalar(
    title="Metrics",
    series="test_rmse_original",
    value=rmse_original,
    iteration=0
    )

    # Закрываем задачу ClearML
if clearml_task:
  clearml_task.close()

Исходная форма телеметрии: (259883, 33), столбцы: ['Время', 'Мощность МПСИ 1', 'Мощность МПСИ 2', 'Мощность МШЦ 1', 'Мощность МШЦ 2', 'Ток МПСИ 1', 'Ток МПСИ 2', 'Ток МШЦ 1', 'Ток МШЦ 2', 'Питание МПСИ 1', 'Питание МПСИ 2', 'Возврат руды МПСИ 1', 'Возврат руды МПСИ 2', 'Расход воды МПСИ 1 PV', 'Расход воды МПСИ 2 PV', 'Расход воды МПСИ 1 SP', 'Расход воды МПСИ 2 SP', 'Расход воды МПСИ 1 CV', 'Расход воды МПСИ 2 CV', 'факт соотношение руда/вода МПСИ 1', 'факт соотношение руда/вода МПСИ 2', 'Давление на подшипник МПСИ 1 загрузка', 'Давление на подшипник МПСИ 2 загрузка', 'Давление на подшипник МПСИ 1 разгрузка', 'Давление на подшипник МПСИ 2 разгрузка', 'Расход оборотной воды 1', 'Расход оборотной воды 2', 'pH оборотной воды', 't оборотной воды', 'Гранулометрия 1', 'Гранулометрия 2', 'Поток 1', 'Поток 2']
После установки индекса (телеметрия): (259883, 32)
После удаления целей: (259883, 30), столбцы: ['Мощность МПСИ 1', 'Мощность МПСИ 2', 'Мощность МШЦ 1', 'Мощность МШЦ 2', 'Ток МПСИ 1', 

Обработка батчей: 100%|██████████| 203/203 [03:32<00:00,  1.05s/it]


Примеры предсказаний в исходных единицах:
Пример 1: Предсказание = [81.61056 82.97548], Истинное значение = [77.004906 77.87027 ]
Пример 2: Предсказание = [81.605   82.53992], Истинное значение = [77.00162 78.7395 ]
Пример 3: Предсказание = [81.5158  82.22056], Истинное значение = [77.00121 78.88742]
Пример 4: Предсказание = [81.31159 82.19593], Истинное значение = [77.0073  78.89084]
Пример 5: Предсказание = [81.168724 82.02257 ], Истинное значение = [77.00843 78.77817]

RMSE в исходных единицах на всём тестовом наборе: 7.775886372273692
2025-03-30 22:16:41,608 - clearml.reporter - WARNING - Event reporting sub-process lost, switching to thread based reporting



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but QuantileTransformer was fitted with feature names

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but QuantileTransformer was fitted with feature names



KeyboardInterrupt: 